# Support Vector Machine



In [1]:
import os, sys, shutil
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:


FOLDER = "/content/drive/MyDrive/Member3_ML_project"

if not os.path.isdir(FOLDER):
    print("Folder not found:", FOLDER)
    print("\nFolders in your MyDrive:")
    for d in sorted(os.listdir("/content/drive/MyDrive")):
        print("   ", d)
    raise SystemExit("Fix FOLDER above, then re-run this cell.")

print("Files in that folder:", os.listdir(FOLDER))

shutil.copy(f"{FOLDER}/rice_shared.py", "/content/rice_shared.py")
sys.path.insert(0, "/content")

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import rice_shared as rs
from sklearn.model_selection import GridSearchCV
sns.set_theme(style="whitegrid")

rs.DATA_PATH = f"{FOLDER}/Rice_Cammeo_Osmancik.arff"
X_train, X_test, y_train, y_test = rs.get_split("all")

print("\nTrain:", X_train.shape, " Test:", X_test.shape)
print("Train balance:", round(y_train.mean(), 4))
print("\nMUST read (3048, 10) and 0.4278 - if not, stop and tell Member 1.")

Files in that folder: ['Rice_Cammeo_Osmancik.arff', 'rice_shared.py']

Train: (3048, 10)  Test: (762, 10)
Train balance: 0.4278

MUST read (3048, 10) and 0.4278 - if not, stop and tell Member 1.


## Tuning
**SVMs must be scaled** — `scale=True`. The RBF kernel measures distance, and `Area` spans tens of thousands of pixels while `Extent` sits between 0 and 1. Unscaled, Area would dominate every distance.



In [3]:
from sklearn.svm import SVC

grid = {"model__C": [0.1, 1, 10, 100],
        "model__gamma": ["scale", 0.01, 0.1],
        "model__kernel": ["rbf"]}

gs = GridSearchCV(rs.make_pipeline(SVC(probability=True, random_state=rs.RANDOM_STATE), scale=True),
                  grid, cv=rs.INNER_CV, scoring="roc_auc", n_jobs=-1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
print(f"Inner-CV ROC-AUC: {gs.best_score_:.4f}   <- tuning score, NOT reported")

Best params: {'model__C': 10, 'model__gamma': 0.01, 'model__kernel': 'rbf'}
Inner-CV ROC-AUC: 0.9799   <- tuning score, NOT reported


## Reported performance
Repeated stratified CV, 10 splits × 3 repeats.

In [4]:
best = gs.best_estimator_.named_steps["model"]

row, res = rs.score(best, X_train, y_train, name="Support Vector Machine", scale=True)
display(pd.DataFrame([{k: v for k, v in row.items()
                       if k.endswith(("_mean", "_std")) or k == "model"}]).round(4).T)

,0
model,Support Vector Machine
accuracy_mean,0.9281
accuracy_std,0.011
balanced_accuracy_mean,0.9258
balanced_accuracy_std,0.0111
precision_mean,0.9223
precision_std,0.0211
recall_mean,0.9093
recall_std,0.0196
f1_mean,0.9155


In [5]:
import os
os.makedirs(f"{FOLDER}/results", exist_ok=True)
rs.save_results(row, res, "results_svm.csv")
shutil.copy("results_svm.csv",       f"{FOLDER}/results/")
shutil.copy("results_svm_folds.npy", f"{FOLDER}/results/")
print("\nSaved to", f"{FOLDER}/results/")
print("Send BOTH files to Member 3 - he needs them for Notebook 4.")

saved: results_svm.csv

Saved to /content/drive/MyDrive/Member3_ML_project/results/
Send BOTH files to Member 3 - he needs them for Notebook 4.


### Seed stability
Does the result survive a different partition? Cheap, and the rubric rewards it.

In [6]:
for seed in [42, 7, 2024]:
    Xtr, _, ytr, _ = rs.get_split("all", seed=seed)
    r, _ = rs.score(best, Xtr, ytr, name=f"seed={seed}", scale=True)
    print(f"seed {seed}: acc {r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f} | "
          f"AUC {r['roc_auc_mean']:.4f}")

seed 42: acc 0.9281 ± 0.0110 | AUC 0.9799
seed 7: acc 0.9289 ± 0.0134 | AUC 0.9801
seed 2024: acc 0.9329 ± 0.0120 | AUC 0.9799


## Feature importance
An RBF-kernel SVM has no coefficients to read, so permutation importance is the only
route. This is itself a reportable trade-off: the SVM may match logistic regression on
accuracy, but it cannot tell the mill *which measurements matter* nearly as directly.

In [7]:
from sklearn.inspection import permutation_importance

svm_fit = rs.make_pipeline(best, scale=True).fit(X_train, y_train)
pi = permutation_importance(svm_fit, X_train, y_train, n_repeats=10,
                            random_state=42, n_jobs=-1, scoring="roc_auc")
display(pd.DataFrame({"feature": X_train.columns,
                      "permutation": pi.importances_mean,
                      "std": pi.importances_std})
        .sort_values("permutation", ascending=False).round(4))

,feature,permutation,std
7,Aspect_Ratio,0.0191,0.0016
2,Major_Axis_Length,0.0161,0.0018
4,Eccentricity,0.0142,0.0015
5,Convex_Area,0.0115,0.0011
0,Area,0.0107,0.0010
9,Solidity,0.0061,0.0007
8,Roundness,0.0060,0.0009
1,Perimeter,0.0025,0.0006
3,Minor_Axis_Length,0.0002,0.0001
6,Extent,0.0002,0.0002


### Is the boundary actually non-linear?
If a linear kernel matches RBF, the classes are linearly separable and the extra
complexity buys nothing — a clean finding for the comparison.

In [8]:
from sklearn.svm import SVC

for k in ["linear", "rbf"]:
    r, _ = rs.score(SVC(kernel=k, C=best.C, random_state=rs.RANDOM_STATE),
                    X_train, y_train, name=k, scale=True)
    print(f"{k:8s} acc={r['accuracy_mean']:.4f} ± {r['accuracy_std']:.4f} | "
          f"AUC={r['roc_auc_mean']:.4f}")

linear   acc=0.9295 ± 0.0114 | AUC=0.9796
rbf      acc=0.9266 ± 0.0120 | AUC=0.9664
